# E-Commerce & Financial Fraud Detection Pipeline
### End-to-End Analysis: EDA, Behavioral Feature Engineering, Hybrid ML (Isolation Forest + Random Forest), and Real-Time Stream Scoring

**Objective:** Detect fraudulent financial transactions combining unsupervised anomaly detection with supervised classification, benchmark near-real-time streaming inference, and quantify business ROI.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to Python path for modular imports
sys.path.insert(0, str(Path.cwd().parent / "src"))

from fraud_detection.config import AppConfig
from fraud_detection.data.loader import DataLoader
from fraud_detection.data.preprocessor import DataPreprocessor
from fraud_detection.features.engineering import FeatureEngineer
from fraud_detection.models.isolation_forest import IsolationForestDetector
from fraud_detection.models.random_forest import RandomForestDetector
from fraud_detection.models.ensemble import HybridFraudEnsemble
from fraud_detection.models.evaluate import evaluate_model, print_evaluation_summary
from fraud_detection.streaming.simulator import StreamSimulator

sns.set_theme(style="whitegrid")
print("Environment initialized successfully.")

## 1. Dataset Loading & Inspection
The pipeline automatically detects if the full PaySim dataset (`data/raw/` or project root) is available; otherwise, it seamlessly utilizes the built-in stratified sample (`data/sample/sample_transactions.csv`).

In [ ]:
config = AppConfig()
data_path = config.data.active_data_path
print(f"Active dataset path: {data_path}")

loader = DataLoader(data_path)
# Load up to 50,000 rows for rapid exploration
df = loader.load(nrows=50000)
df.head()

In [ ]:
print("Dataset Shape:", df.shape)
print("Class Distribution (isFraud):")
print(df['isFraud'].value_counts(normalize=True))
df.describe()

## 2. Exploratory Data Analysis & Visualizations

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='type', hue='isFraud', palette=['#3498db', '#e74c3c'])
plt.title("Transaction Frequency by Type and Fraud Status")
plt.yscale('log')
plt.ylabel("Count (log scale)")
plt.show()

## 3. Behavioral Feature Engineering & Stratified Split

In [ ]:
preprocessor = DataPreprocessor(test_size=0.2, random_state=42)
df_encoded = preprocessor.encode_transaction_types(df)
train_df, test_df = preprocessor.split_data(df_encoded, target_col='isFraud')

engineer = FeatureEngineer()
train_feat = engineer.fit_transform(train_df)
test_feat = engineer.transform(test_df)

feature_cols = config.features.model_features
X_train, y_train = engineer.extract_feature_matrix(train_feat, feature_cols)
X_test, y_test = engineer.extract_feature_matrix(test_feat, feature_cols)

print("Training feature matrix shape:", X_train.shape)
print("Test feature matrix shape:", X_test.shape)

## 4. Model Training & Hybrid Ensemble
We train:
1. **Isolation Forest** (Unsupervised anomaly detector for zero-day fraud patterns)
2. **Random Forest** (Supervised cost-sensitive classifier with balanced class weights)
3. **Hybrid Ensemble** (Multi-layered defense-in-depth architecture)

In [ ]:
iso = IsolationForestDetector(n_estimators=100, contamination=0.01, random_state=42)
rf = RandomForestDetector(n_estimators=50, max_depth=10, class_weight='balanced', random_state=42)
ensemble = HybridFraudEnsemble(strategy='union', rf_detector=rf, iso_detector=iso)

print("Fitting Hybrid Ensemble...")
ensemble.fit(X_train, y_train)
print("Model training complete.")

## 5. Evaluation & Business Impact Quantification

In [ ]:
test_amounts = test_df['amount']
y_pred = ensemble.predict(X_test)
y_proba = ensemble.predict_proba(X_test)

metrics = evaluate_model(y_test, y_pred, y_proba=y_proba, amounts=test_amounts)
print_evaluation_summary(metrics, title="Hybrid Ensemble Evaluation")

## 6. Near-Real-Time Stream Processing Simulation

In [ ]:
simulator = StreamSimulator(model=ensemble, feature_columns=feature_cols)
summary = simulator.simulate(test_feat, delay_seconds=0.005, max_events=20)